# 高精度OCRパイプライン
## 画像前処理 + 領域分割 + 複数OCRエンジン

## 1. ライブラリインポート

In [ ]:
import cv2
import numpy as np
import easyocr
import json
from pathlib import Path
from collections import Counter
from typing import List, Tuple, Dict
import pandas as pd
from IPython.display import display, Image as IPImage
import matplotlib.pyplot as plt
from matplotlib import rcParams

# 日本語フォント設定
rcParams['font.sans-serif'] = ['DejaVu Sans']

print("✅ ライブラリインポート完了")

## 2. 修正ルール辞書

In [ ]:
CORRECTION_RULES = {
    'ニャンギラス': ['ニャンギラス', 'ニャンギラ', 'ニャン'],
    'マギレコ': ['マギレコ', 'マギ'],
    'モンハンライズ': ['モンハンライズ', 'モンハン'],
    'バイオ5': ['バイオ5', 'バイオ'],
    'ジャグラー': ['ジャグラー', 'ジャグ'],
    'からくり': ['からくり', 'からく'],
    'アズールレーン': ['アズールレーン', 'アズール'],
    'スタディライト': ['スタディライト', 'スタデ'],
    'シャーマシング': ['シャーマシング', 'シャー'],
    'ゴーギャッシュ': ['ゴーギャッシュ', 'ゴー'],
    '沖ドキ': ['沖ドキ', '沖'],
    'キンハナ': ['キンハナ', 'キン'],
    'エウレカ': ['エウレカ', 'エウ'],
    'カバネリ': ['カバネリ', 'カバ'],
    'デビルメイクライ': ['デビルメイクライ', 'デビル'],
    'TOL.LOVE6': ['TOL.LOVE6', 'TOL'],
    'ガリラ': ['ガリラ', 'ガリ'],
    'ディスク2': ['ディスク2', 'ディス'],
    'バンドリ': ['バンドリ', 'パンドリ'],
    'SAO': ['SAO', 'ＳＡＯ'],
    'SBJ': ['SBJ', 'ＳＢＪ'],
    'AT': ['AT', 'ＡＴ'],
    'DMC': ['DMC', 'ＤＭＣ'],
    '本日': ['本日'],
    '水曜日': ['水曜日', '水'],
    '木曜日': ['木曜日', '木'],
    '金曜日': ['金曜日', '金'],
    '土曜日': ['土曜日', '土'],
    '日曜日': ['日曜日', '日'],
    '月曜日': ['月曜日', '月'],
}

print(f"✅ {len(CORRECTION_RULES)}個の修正ルール")

## 3. ユーティリティ関数

In [ ]:
def preprocess_image(image_path: str, upscale: int = 2, denoise: int = 10):
    """画像を前処理"""
    print("🖼️  画像を読み込み中...")
    img = cv2.imread(image_path)
    print(f"   元の解像度: {img.shape[1]}x{img.shape[0]}")
    
    print("  [1/6] グレースケール...")
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    
    print(f"  [2/6] ノイズ除去...")
    denoised = cv2.fastNlMeansDenoising(gray, None, h=denoise, templateWindowSize=7, searchWindowSize=21)
    
    print("  [3/6] コントラスト強化 (CLAHE)...")
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    enhanced = clahe.apply(denoised)
    
    print("  [4/6] シャープニング...")
    kernel = np.array([[-1,-1,-1], [-1,9,-1], [-1,-1,-1]])
    sharpened = cv2.filter2D(enhanced, -1, kernel)
    
    print("  [5/6] 二値化...")
    _, binary = cv2.threshold(sharpened, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    
    print(f"  [6/6] 高解像度化 ({upscale}x)...")
    upscaled = cv2.resize(binary, None, fx=upscale, fy=upscale, interpolation=cv2.INTER_CUBIC)
    
    print(f"   処理後解像度: {upscaled.shape[1]}x{upscaled.shape[0]}")
    
    return upscaled, img, binary


def split_vertical(image: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    """画像を左右に分割"""
    h, w = image.shape[:2]
    mid = w // 2
    return image[:, :mid], image[:, mid:]


def correct_text(text: str, rules: Dict) -> str:
    """テキストを修正"""
    for correct_word, error_patterns in rules.items():
        for pattern in error_patterns:
            if pattern in text:
                text = text.replace(pattern, correct_word)
    return text


def analyze_results(results: List[Tuple]) -> Dict:
    """結果を分析"""
    texts = [item[1] for item in results]
    confidences = [item[2] for item in results if isinstance(item[2], float)]
    low_confidence = [(text, conf) for (_, text, conf) in results 
                      if isinstance(conf, float) and conf < 0.7]
    
    return {
        'total_lines': len(results),
        'avg_confidence': sum(confidences) / len(confidences) if confidences else 0,
        'min_confidence': min(confidences) if confidences else 0,
        'max_confidence': max(confidences) if confidences else 0,
        'low_confidence_count': len(low_confidence),
        'low_confidence_details': low_confidence,
        'word_frequency': Counter(texts).most_common(30),
    }

print("✅ ユーティリティ関数定義完了")

## 4. 画像パスの指定

In [ ]:
# 📝 画像ファイルのパスを指定
image_path = "path/to/your/image.png"  # <- 編集: 画像のパスを入力

image_file = Path(image_path)
if image_file.exists():
    print(f"✅ 画像ファイル: {image_file}")
    print(f"   サイズ: {image_file.stat().st_size / 1024:.1f} KB")
else:
    print(f"❌ エラー: {image_path} が見つかりません")

## 5. ステップ1: 画像前処理

In [ ]:
print("\n【ステップ1】画像前処理")
print("-" * 70)

processed_image, original_image, binary_image = preprocess_image(
    image_path, 
    upscale=2,
    denoise=10
)

### 前処理の可視化

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].imshow(cv2.cvtColor(original_image, cv2.COLOR_BGR2RGB))
axes[0].set_title('元の画像')
axes[0].axis('off')

axes[1].imshow(binary_image, cmap='gray')
axes[1].set_title('二値化')
axes[1].axis('off')

axes[2].imshow(processed_image, cmap='gray')
axes[2].set_title('アップスケール済み')
axes[2].axis('off')

plt.tight_layout()
plt.show()

print("✅ 前処理完了")

## 6. ステップ2: 領域分割

In [ ]:
print("\n【ステップ2】領域分割")
print("-" * 70)
print("🔀 画像を左右に分割...")

left_image, right_image = split_vertical(processed_image)

print(f"   左領域: {left_image.shape[1]}x{left_image.shape[0]}")
print(f"   右領域: {right_image.shape[1]}x{right_image.shape[0]}")

# 可視化
fig, axes = plt.subplots(1, 2, figsize=(12, 8))

axes[0].imshow(left_image, cmap='gray')
axes[0].set_title('左領域')
axes[0].axis('off')

axes[1].imshow(right_image, cmap='gray')
axes[1].set_title('右領域')
axes[1].axis('off')

plt.tight_layout()
plt.show()

## 7. ステップ3: EasyOCR実行

In [ ]:
print("\n【ステップ3】OCR処理")
print("-" * 70)
print("🔧 EasyOCRを初期化中... (GPU: True)")

reader = easyocr.Reader(['ja'], gpu=True)
print("✅ EasyOCR初期化完了\n")

# 左領域
import tempfile
print("🔍 左領域をOCR...")
with tempfile.NamedTemporaryFile(suffix='.png', delete=False) as f:
    cv2.imwrite(f.name, left_image)
    left_results = reader.readtext(f.name)
print(f"   {len(left_results)}行を検出")

# 右領域
print("🔍 右領域をOCR...")
with tempfile.NamedTemporaryFile(suffix='.png', delete=False) as f:
    cv2.imwrite(f.name, right_image)
    right_results = reader.readtext(f.name)
print(f"   {len(right_results)}行を検出")

# マージ
all_results = left_results + right_results
print(f"\n✅ 合計 {len(all_results)}行を検出")

## 8. ステップ4: 結果分析

In [ ]:
print("\n【ステップ4】結果分析")
print("-" * 70)

analysis = analyze_results(all_results)

print(f"検出行数: {analysis['total_lines']}")
print(f"平均信頼度: {analysis['avg_confidence']:.2%}")
print(f"信頼度範囲: {analysis['min_confidence']:.2%} - {analysis['max_confidence']:.2%}")
print(f"信頼度70%未満: {analysis['low_confidence_count']}件\n")

if analysis['low_confidence_details']:
    print("⚠️  信頼度が低いテキスト (トップ10):")
    for text, conf in sorted(analysis['low_confidence_details'], 
                             key=lambda x: x[1])[:10]:
        print(f"  {conf:.2%}: '{text}'")

print("\n📈 単語出現頻度 (トップ15):")
for word, count in analysis['word_frequency'][:15]:
    print(f"  {count:3d}回: '{word}'")

## 9. ステップ5: テキスト修正

In [ ]:
print("\n【ステップ5】テキスト修正")
print("-" * 70)

# 修正前後の比較
comparison_data = []
for (bbox, text, confidence) in all_results:
    if confidence < 0.8:
        corrected = correct_text(text, CORRECTION_RULES)
        if text != corrected:
            comparison_data.append({
                '信頼度': f"{confidence:.1%}",
                '修正前': text,
                '修正後': corrected
            })

if comparison_data:
    print("修正されたテキスト:")
    df = pd.DataFrame(comparison_data)
    display(df)
else:
    print("修正対象はありません")

## 10. ステップ6: 結果保存

In [ ]:
print("\n【ステップ6】結果保存")
print("-" * 70)

base_name = Path(image_path).stem

# 修正後テキスト
corrected_lines = []
for (bbox, text, confidence) in all_results:
    corrected = correct_text(text, CORRECTION_RULES)
    corrected_lines.append(corrected)

corrected_text = '\n'.join(corrected_lines)

# テキスト保存
output_txt = f"{base_name}_ocr_high_precision.txt"
with open(output_txt, 'w', encoding='utf-8') as f:
    f.write(corrected_text)
print(f"✅ {output_txt} を保存")

# JSON保存
output_json = f"{base_name}_ocr_high_precision.json"
json_data = []
for (bbox, text, confidence) in all_results:
    corrected = correct_text(text, CORRECTION_RULES)
    json_data.append({
        'text': corrected,
        'confidence': float(confidence),
        'bbox': [[float(coord) for coord in point] for point in bbox] if bbox else None
    })

with open(output_json, 'w', encoding='utf-8') as f:
    json.dump(json_data, f, ensure_ascii=False, indent=2)
print(f"✅ {output_json} を保存")

print("\n✨ 処理完了！")

## 11. 全テキスト確認（オプション）

In [ ]:
print("✨ 修正後のテキスト（全行）:")
print("="*70)
print(corrected_text)